# 05 · Validation Plan — fibril-vs-monomer ELISA/SPR + controls + diagnostic framing

**Standard slot:** *validation plan.* **For Project 11 this means:** turn the fibril-selective top
candidates into a **costed, controlled wet-lab plan** whose centerpiece is a **fibril-vs-monomer**
selectivity assay (ELISA/SPR), with the mandatory controls (positive conformational antibody,
**scrambled-interface** negative, monomer/unrelated negatives), an expression strategy, and the
**diagnostic-tracer** framing (PET tracer / assay) (D4/D5).

A design that passes every filter — even the in-silico monomer counter-test — is a **hypothesis**. The
**fibril-vs-monomer assay** is what tests it. Needs `results/top_candidates.csv` (notebook 04).

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Draft the experimental validation plan

Generate a plan card from the top candidates: the fibril-vs-monomer assay, controls, expression,
timeline, costed reagents, and the diagnostic-tracer framing. Fill the `<...>` from your own numbers;
this is the deliverable other people will actually read.

In [ ]:
import pandas as pd, os

top = pd.read_csv("results/top_candidates.csv") if os.path.exists("results/top_candidates.csv") else pd.DataFrame()
n_top = len(top)
by_par = top.groupby("paradigm").size().to_dict() if n_top else {}

plan = f"""# Conformation-Specific Fibril-Binder Validation Plan (Project 11 — by <your name>, <date>)

## Candidates
Top {n_top} FIBRIL-SELECTIVE candidates carried forward ({by_par}); see results/top_candidates.csv.
EVERY in-silico number is a HYPOTHESIS until measured — `pae_interaction` is confidence (not affinity),
and the `specificity_gap` is a model proxy (not a measured fold-selectivity). The monomer is
disordered, so its model is itself uncertain. The fibril-vs-monomer assay below is the real test.

## Target conformations (the whole point)
- ON-target: the AMYLOID FIBRIL (tau PHF from 5O3L/5O3T, or alpha-synuclein fibril from 6CU7/6H6B).
  Prepare recombinant fibrils in vitro (seeded aggregation); confirm fibrils by ThT fluorescence + TEM/cryo-EM.
- OFF-target (counter): the MONOMER of the SAME protein (freshly purified, kept monomeric; verify by SEC).
- OFF-target (cross-amyloid): the OTHER amyloid fibril (tau vs alpha-syn) for diagnostic discrimination.

## Expression strategy
- Binders: E. coli BL21(DE3), His-tagged, 16-18 C overnight; IMAC + SEC. Small (50-90 aa) -> high yield expected.
- Antigens: recombinant tau / alpha-synuclein; prepare BOTH a monomer prep AND an in-vitro fibril prep
  from the same construct (this matched pair is what makes the selectivity readout clean).

## Assays (go/no-go -> basic -> the selectivity test)
1. Go/no-go: express binder -> SDS-PAGE -> SEC (monodisperse?).
2. Binding: SPR or BLI vs immobilized FIBRIL -> apparent K_D + kinetics. Test a dilution series.
3. THE SELECTIVITY TEST (the point): fibril-vs-monomer ELISA/SPR -> signal on FIBRIL must be >> signal
   on MONOMER (report the selectivity RATIO measured here; do NOT report the in-silico gap as the result).
4. Cross-amyloid: same readout vs the OTHER amyloid fibril (must be low for a discriminating tracer).
5. (Diagnostic deep dive) tissue staining / fibril pulldown from patient-derived material under approval.

## Controls (MANDATORY)
- Positive: a known conformation-specific anti-fibril antibody/tracer (e.g., a conformational mAb or a
  validated amyloid PET-tracer scaffold) -> assay + fibril prep are active and conformation-discriminating.
- Negative (scrambled-interface): YOUR OWN top design with its fibril-contacting residues scrambled
  -> must LOSE fibril binding (cleanest specificity control).
- Negative (monomer): the MONOMER of the same protein -> a selective binder must NOT bind it.
- Negative (unrelated): an unrelated mini-protein / an unrelated amyloid -> should not bind.

## Diagnostic-tracer framing
The intended use is DIAGNOSTIC (a conformation-selective probe for PET imaging or a fibril-detection
assay) and/or an aggregation MODULATOR — recognizing pathological aggregates, not the physiological
monomer. This is a defensible, in-scope neurodegeneration application (low dual-use). A clinical PET
tracer additionally needs BBB penetration, radiolabeling chemistry, and pharmacokinetics — out of
scope for this capstone but named here as the translational path.

## Realistic expectations
Conformational selectivity is VERY hard: the binder must REJECT the abundant monomer. The MAJORITY of
in-silico "selective" designs will fail the monomer counter-test experimentally. Report the measured
selectivity ratio and the experimental hit rate honestly. Do NOT imply a working tracer or fabricate
a K_D / selectivity number.

## Timeline + costed reagents (fill in)
- Gene synthesis ({n_top} binders + scrambled-interface negatives): $<...>, <...> weeks (IGSC-screened provider).
- Recombinant tau / alpha-syn (monomer + fibril preps) + ThT + TEM time + SPR/BLI chips + positive-control mAb: $<...>.
- Personnel/instrument time: <...> weeks.

## Responsible research
Conformation-selective binders to pathological amyloid aggregates for neurodegeneration DIAGNOSTICS /
aggregation modulation (in scope; low dual-use). Gene synthesis via a biosecurity-screening provider;
any patient-derived material + wet lab under institutional biosafety/ethics approval.
"""
os.makedirs("results", exist_ok=True)
open("results/validation_plan.md", "w").write(plan)
print("wrote results/validation_plan.md — fill the <...> placeholders from your numbers.")
print(plan[:700], "...")

## 2 · Build the scrambled-interface negative controls

The single cleanest specificity control: take each top design and **scramble its fibril-contacting
residues** — it should **lose** fibril binding. Generating these alongside the real designs (same
expression batch) makes the fibril-vs-monomer comparison airtight. Here we scaffold the sequence-level
scramble deterministically; on Colab, scramble the *interface* positions specifically using the
predicted fibril contacts.

In [ ]:
import random
import binder_tools as bt   # bt._hashints gives a DETERMINISTIC seed (Python's hash() is salted)

def scramble_interface(seq, frac=0.4, seed=0):
    """Deterministically shuffle a fraction of the sequence as a NEGATIVE-CONTROL stand-in.
    On Colab, scramble the predicted FIBRIL-INTERFACE residues specifically (positions contacting the fibril)."""
    rng = random.Random(seed)
    seq = list(seq)
    idx = list(range(len(seq)))
    rng.shuffle(idx)
    k = max(1, int(len(seq) * frac))
    chosen = idx[:k]
    vals = [seq[i] for i in chosen]
    rng.shuffle(vals)
    for i, v in zip(chosen, vals):
        seq[i] = v
    return "".join(seq)

negs = []
if n_top and "sequence" in top.columns:
    for _, r in top.iterrows():
        s = str(r.get("sequence", ""))
        if s and s.lower() != "nan":
            negs.append(dict(design_id=str(r["design_id"]) + "_SCRAM",
                             parent=r["design_id"], paradigm=r.get("paradigm"),
                             sequence=scramble_interface(s, seed=bt._hashints(r["design_id"]) % 10**6),
                             role="scrambled-interface negative control"))
    pd.DataFrame(negs).to_csv("results/negative_controls.csv", index=False)
    print(f"wrote results/negative_controls.csv: {len(negs)} scrambled-interface negatives")
else:
    print("Run notebook 04 first to produce results/top_candidates.csv with sequences (need fibril-selective hits).")

## 3 · (Stretch) Boltz-2 affinity on top hits `[stretch]`

Boltz-2 can predict a binding-affinity signal for the top complexes. Use it for **relative ranking +
caveats only** — **never fabricate a K_D**, and never present a predicted number as measured. For a
fibril target this is even less reliable (large, repetitive assembly), so treat it as a tie-breaker for
which selective hits to test first, not as evidence of binding or of selectivity.

In [ ]:
# Scaffold ONLY. Do NOT invent affinities or selectivity ratios. On Colab:
#   pip install boltz; build the (binder, fibril) complex input; run boltz predict with affinity mode;
#   read the predicted-affinity signal and report the RELATIVE ranking of the top hits + heavy caveats.
# Pinned upstream (verify): https://github.com/jwohlwend/boltz
print("Boltz-2 affinity is a STRETCH scaffold: relative ranking + caveats only, NEVER a fabricated K_D.")
print("Use it to PRIORITIZE which fibril-selective hits to test first in the assay — not as evidence of binding.")

## D4 / D5 checklist
- [ ] `results/validation_plan.md` completed: **fibril-vs-monomer** ELISA/SPR (the selectivity test), cross-amyloid readout, expression, timeline, costed reagents.
- [ ] Controls specified: positive (conformational anti-fibril antibody/tracer), **scrambled-interface** negative (`results/negative_controls.csv`), **monomer** negative, unrelated negative.
- [ ] Matched monomer + in-vitro-fibril preps planned (ThT + TEM/cryo-EM confirmation).
- [ ] Diagnostic-tracer framing stated (PET / assay; BBB + radiochemistry named as the translational path).
- [ ] (Stretch) Boltz-2 affinity used only for relative ranking, with caveats — no fabricated K_D / selectivity.
- [ ] Honest framing: every design is a hypothesis until the fibril-vs-monomer assay; report the measured selectivity ratio + experimental hit rate.
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release.

You're done — a conformation-specific fibril binder set with a fibril-vs-monomer validation plan and a diagnostic-tracer framing.